In [ ]:
%%bash
# Cell 1: تنزيل المشروع وتجهيز المجلدات (Kaggle)
set -euo pipefail

PROJECT_DIR="/kaggle/working/NAMAA-Egyptian-Voice"
REPO_URL="https://github.com/kareemkamal10/NAMAA-Egyptian-Voice.git"

if [ ! -d "${PROJECT_DIR}/.git" ]; then
  git clone "${REPO_URL}" "${PROJECT_DIR}"
else
  git -C "${PROJECT_DIR}" pull --ff-only
fi

git -C "${PROJECT_DIR}" lfs install
git -C "${PROJECT_DIR}" lfs pull

mkdir -p "${PROJECT_DIR}/outputs" "${PROJECT_DIR}/uploads" "${PROJECT_DIR}/logs" "${PROJECT_DIR}/runtime"
echo "Download step completed: ${PROJECT_DIR}"

In [ ]:
%%bash
# Cell 2: إعداد البيئة وتثبيت المتطلبات (Kaggle)
set -euo pipefail

PROJECT_DIR="/kaggle/working/NAMAA-Egyptian-Voice"
VENV_DIR="/kaggle/working/namaa-venv"

python3 "${PROJECT_DIR}/scripts/setup_env.py" \
  --project-dir "${PROJECT_DIR}" \
  --venv-path "${VENV_DIR}" \
  --venv-backend auto \
  --log-name ceatevenv

echo "Environment setup completed."

In [ ]:
# Cell 3: تشغيل المشروع كخدمة خلفية (Kaggle)
from pathlib import Path
import json
import os
import signal
import subprocess
import time
import urllib.request

CELL_REV = \"2026-04-20-kaggle-r2\"\nPROJECT_DIR = Path("/kaggle/working/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/kaggle/working/namaa-venv")
RUNTIME_DIR = PROJECT_DIR / "runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_HOST = "127.0.0.1"
SERVICE_PORT = 7861
IDLE_TIMEOUT_SECONDS = 600
SERVICE_URL = f"http://{SERVICE_HOST}:{SERVICE_PORT}"
SERVICE_STATE_FILE = RUNTIME_DIR / "model_service_state.json"
SERVICE_LOG_FILE = PROJECT_DIR / "logs" / "model_service.log"

print(f"Cell 3 revision: {CELL_REV}")

if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

if not VENV_PYTHON.exists():
    raise FileNotFoundError("Virtualenv python not found. Run Cell 2 first.")
if not (PROJECT_DIR / "app.py").exists():
    raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")


def is_pid_alive(pid):
    if pid is None:
        return False
    try:
        os.kill(int(pid), 0)
        return True
    except Exception:
        return False


def check_service_health(url):
    try:
        with urllib.request.urlopen(f"{url}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False


existing_pid = None
if SERVICE_STATE_FILE.exists():
    try:
        existing_pid = int(json.loads(SERVICE_STATE_FILE.read_text(encoding="utf-8")).get("pid", 0))
    except Exception:
        existing_pid = None

if is_pid_alive(existing_pid) and check_service_health(SERVICE_URL):
    print(f"Service already running: {SERVICE_URL} (pid={existing_pid})")
else:
    if is_pid_alive(existing_pid):
        try:
            os.kill(existing_pid, signal.SIGTERM)
        except Exception:
            pass

    SERVICE_LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    log_handle = SERVICE_LOG_FILE.open("a", encoding="utf-8")

    command = [
        str(VENV_PYTHON),
        "app.py",
        "serve",
        "--host",
        SERVICE_HOST,
        "--port",
        str(SERVICE_PORT),
        "--idle-timeout",
        str(IDLE_TIMEOUT_SECONDS),
    ]
    process = subprocess.Popen(
        command,
        cwd=str(PROJECT_DIR),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

    ready = False
    for _ in range(180):
        if process.poll() is not None:
            break
        if check_service_health(SERVICE_URL):
            ready = True
            break
        time.sleep(1)

    if not ready:
        tail = ""
        if SERVICE_LOG_FILE.exists():
            tail = SERVICE_LOG_FILE.read_text(encoding="utf-8", errors="ignore")[-2500:]
        raise RuntimeError(f"Service failed to start.\n{tail}")

    SERVICE_STATE_FILE.write_text(
        json.dumps(
            {
                "pid": process.pid,
                "url": SERVICE_URL,
                "idle_timeout_seconds": IDLE_TIMEOUT_SECONDS,
                "log_file": str(SERVICE_LOG_FILE),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Service started: {SERVICE_URL} (pid={process.pid})")

globals()["SERVICE_URL"] = SERVICE_URL
globals()["SERVICE_STATE_FILE"] = str(SERVICE_STATE_FILE)
globals()["SERVICE_LOG_FILE"] = str(SERVICE_LOG_FILE)

print(f"Health endpoint: {SERVICE_URL}/health")
print(f"Generate endpoint: {SERVICE_URL}/generate")
print(f"API placeholder: {SERVICE_URL}/api-placeholder")
print(f"Service log file: {SERVICE_LOG_FILE}")
print("Idle timeout: 10 minutes without generation requests.")

In [ ]:
# Cell 4: واجهة Web فوق الخدمة الخلفية (Kaggle)
from pathlib import Path
import json
import logging
import urllib.error
import urllib.request
import subprocess
import sys

CELL_NAME = "cell_04_kaggle_web_ui"
CELL_REV = \"2026-04-20-kaggle-r2\"\nLOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Start Cell 4")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 4 revision: {CELL_REV}")
logger.info(f"Cell 4 revision: {CELL_REV}")


def ensure_gradio():
    try:
        import gradio as gr
        return gr
    except Exception:
        print("Installing Gradio (optional UI layer)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44,<5"], check=True)
        import gradio as gr
        return gr


gr = ensure_gradio()
SERVICE_URL = str(globals().get("SERVICE_URL", "http://127.0.0.1:7861")).rstrip("/")


def get_json(url):
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode("utf-8"))


def post_json(url, payload):
    request = urllib.request.Request(
        url=url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json; charset=utf-8"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=600) as response:
        return json.loads(response.read().decode("utf-8"))


def service_is_ready():
    try:
        health = get_json(f"{SERVICE_URL}/health")
        return bool(health.get("status") == "ok")
    except Exception:
        return False


def run_service_session(prompt, reference_audio, exaggeration, cfg_pace, seed, temperature, outputname):
    prompt_value = str(prompt or "").strip()
    if not prompt_value:
        yield None, "❌ اكتب النص أولًا."
        return

    if not service_is_ready():
        yield None, "❌ الخدمة غير شغالة. شغل Cell 3 أولًا."
        return

    reference_path = ""
    if isinstance(reference_audio, str):
        reference_path = reference_audio.strip()
    elif isinstance(reference_audio, dict):
        reference_path = str(reference_audio.get("path") or "").strip()

    payload = {
        "prompt": prompt_value,
        "reference": reference_path,
        "exaggeration": float(exaggeration),
        "cfg_pace": float(cfg_pace),
        "seed": int(float(seed)),
        "temperature": float(temperature),
        "outputname": str(outputname or "").strip() or "output",
        "output_dir": "outputs",
    }

    yield None, "⏳ جاري التوليد عبر الخدمة الخلفية..."

    try:
        result = post_json(f"{SERVICE_URL}/generate", payload)
    except urllib.error.HTTPError as error:
        body = ""
        try:
            body = error.read().decode("utf-8", errors="ignore")
        except Exception:
            pass
        logger.error(f"HTTP error: {error} | body={body}")
        yield None, f"❌ فشل التوليد: {body or error}"
        return
    except Exception as error:
        logger.exception("Service request failed")
        yield None, f"❌ فشل الاتصال بالخدمة: {error}"
        return

    output_path = str(result.get("output_path") or "").strip()
    if not output_path:
        yield None, "❌ لم يتم استلام مسار الملف الناتج من الخدمة."
        return

    status = (
        f"✅ تم التوليد بنجاح عبر الخدمة.\n\n"
        f"- output: {Path(output_path).name}\n"
        f"- path: {output_path}\n"
        f"- service: {SERVICE_URL}"
    )
    yield output_path, status


custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;800&display=swap');

:root {
  --n-bg: radial-gradient(1200px 520px at 12% 0%, #0f766e22, transparent 60%),
          radial-gradient(1200px 520px at 95% 100%, #0891b222, transparent 60%),
          linear-gradient(135deg, #0b1220 0%, #111827 100%);
  --n-card: #0f172ad9;
  --n-text: #e5e7eb;
  --n-accent: #14b8a6;
  --n-accent-2: #06b6d4;
}

.gradio-container {
  font-family: 'Cairo', sans-serif !important;
  background: var(--n-bg) !important;
  color: var(--n-text) !important;
}

.main-card {
  border: 1px solid #ffffff26;
  background: var(--n-card);
  border-radius: 20px;
  padding: 18px;
  box-shadow: 0 10px 40px #00000040;
}

#run-btn {
  background: linear-gradient(135deg, var(--n-accent), var(--n-accent-2)) !important;
  border: 0 !important;
  color: #001018 !important;
  font-weight: 800 !important;
}

#run-btn:hover {
  filter: brightness(1.06);
}

footer {
  display: none !important;
}
"""

with gr.Blocks(css=custom_css, title="NAMAA Studio - Kaggle") as demo:
    gr.Markdown(
        f"""
        <div class='main-card'>
          <h2 style='margin:0;'>NAMAA Studio</h2>
          <p style='margin:6px 0 0 0; opacity:.9;'>
            واجهة اختيارية تعمل فوق الخدمة الخلفية: {SERVICE_URL}
          </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=5):
            prompt_box = gr.Textbox(
                label="Prompt النص",
                placeholder="اكتب النص الذي تريد تحويله...",
                lines=5,
                value="انا سبت الشغل و راجع دلوقتي علي طول.",
            )
            reference_box = gr.Audio(
                label="Reference صوتي (اختياري)",
                sources=["upload"],
                type="filepath",
            )
            with gr.Row():
                exaggeration_box = gr.Slider(0.25, 2.0, value=0.5, step=0.05, label="exaggeration")
                cfg_pace_box = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="cfg_pace")
            with gr.Row():
                seed_box = gr.Number(value=0, precision=0, label="seed")
                temperature_box = gr.Slider(0.05, 5.0, value=0.8, step=0.05, label="temperature")
            outputname_box = gr.Textbox(value="output", label="outputname")
            run_button = gr.Button("Generate Audio", elem_id="run-btn")

        with gr.Column(scale=5):
            output_audio = gr.Audio(label="Output Audio", type="filepath", autoplay=False)
            status_md = gr.Markdown("✅ جاهز للتشغيل.")

    run_button.click(
        fn=run_service_session,
        inputs=[
            prompt_box,
            reference_box,
            exaggeration_box,
            cfg_pace_box,
            seed_box,
            temperature_box,
            outputname_box,
        ],
        outputs=[output_audio, status_md],
        show_progress="minimal",
    )

launch_result = demo.queue(default_concurrency_limit=1).launch(
    share=True,
    inline=False,
    inbrowser=False,
    prevent_thread_lock=True,
)

share_url = None
if isinstance(launch_result, tuple) and len(launch_result) >= 3:
    share_url = launch_result[2]

if share_url:
    print(f" رابط Kaggle العام: {share_url}")
else:
    print(" لم يتم إنشاء رابط عام. فعّل Internet من إعدادات Kaggle ثم أعد تشغيل Cell 4.")
logger.info("End Cell 4")